# OULAD — AN7: Outcome Associations

**Status:** package prepared; SQL not executed. Run cells individually. Do not use Run All. AN7.04 and AN7.05 are separate high-cost scans and must never run concurrently.

## 1. Purpose and boundaries

AN7 studies associations between recorded VLE activity and retrospective outcomes among registrations eligible at a defined landmark. It is not causal inference or predictive modeling. Primary features use post-registration exposure; course-relative measures remain separate for AN6 reconciliation. final_result is an outcome, never a predictor.

## 2. Frozen contract

- Grain: (code_module, code_presentation, id_student).
- L28 = days 0–27; L56 = 0–55; L84 = 0–83, inclusive.
- O1: Withdrawn vs other known final results among eligible registrations.
- O2: Fail vs Pass+Distinction among eligible non-withdrawn outcomes.
- O3: ordered Fail < Pass < Distinction among eligible non-withdrawn outcomes.
- The 93 Withdrawn/no-date cases have indeterminate landmark eligibility; the nine Fail/with-date cases remain visible for sensitivity.
- Active days are normalized by the number of post-registration days available before each landmark; raw counts and exposure duration remain available for review.
- Resource breadth is reported with exposure duration and requires registration-timing sensitivity because later registration shortens observation time.
- No activity truncation at date_unregistration; no source deduplication.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr
from IPython.display import display, Markdown

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [_current, *_current.parents]
        if (candidate / "src" / "analytics" / "db.py").is_file()
        and (candidate / "sql" / "04_analytics").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.run_an7 import verify_read_only_preflight, run_an7_query, verify_cohort_gate

query_results = {}
KEY = ['code_module', 'code_presentation', 'id_student']
EXPECTED_REGISTRATIONS = 32_593
EXPECTED_DISTINCT_PEOPLE = 28_785
EXPECTED_ACTIVE_REGISTRATIONS = 28_500
EXPECTED_IN_COURSE_ROWS = 9_966_292

## 3. Read-only preflight

AN0.01 verifies database identity and both read-only settings before any AN7 SQL.

In [ ]:
preflight = verify_read_only_preflight()

## 4. AN7.01 — Outcome and registration reconciliation

Inspect all three result sets. Historical 93/9 conflict counts are references, not values to force.

In [ ]:
r = run_an7_query('AN7.01', max_rows=500, display_results=False)
query_results['AN7.01'] = r
for item in r['result_sets']:
    print('Result set', item['statement_number'], '| rows', item['rows_displayed'], '| truncated', item['display_truncated'])
    display(item['data'])

## 5. AN7.02 — Assessment landmark context

Due-date/weight metadata does not prove score or feedback availability. Reuse AN4 saved outputs for submissions and scores.

In [ ]:
r = run_an7_query('AN7.02', max_rows=500, display_results=False)
query_results['AN7.02'] = r
display(r['data'])

## 6. AN7.03 — Landmark cohort eligibility

This one-row-per-student-presentation result retains missing registration dates and undated Withdrawn cases as explicit statuses. The gate must pass before either heavy scan.

In [ ]:
r = run_an7_query('AN7.03', max_rows=33_000, display_results=False)
query_results['AN7.03'] = r
display(r['data'].head())
verify_cohort_gate(r)

## 7. AN7.04 — Daily activity extraction (high cost)

Run alone. Daily pre-aggregation reduces source rows to student-presentation × day before counting active days.

In [ ]:
r = run_an7_query('AN7.04', max_rows=33_000, display_results=False)
query_results['AN7.04'] = r
assert not r['display_truncated'], 'AN7.04 truncated; stop.'
display(r['data'].head())
print('Rows:', len(r['data']), 'Elapsed seconds:', r['elapsed_seconds'])

## 8. Manual review gate — AN7.04

Inspect the complete AN7.04 frame and its retrieval metadata before allowing the second high-cost scan. This gate is intentionally manual; a successful SQL return alone is not analytical approval.

In [ ]:
AN7_04_REVIEWED = False
# After reviewing the complete AN7.04 output and its diagnostics, change this to True.
print('AN7.04 manual review approved:', AN7_04_REVIEWED)

In [ ]:
assert AN7_04_REVIEWED, 'Review the complete AN7.04 output and set AN7_04_REVIEWED = True in the preceding review cell.'
r = run_an7_query('AN7.05', max_rows=33_000, display_results=False)
query_results['AN7.05'] = r
assert not r['display_truncated'], 'AN7.05 truncated; stop.'
display(r['data'].head())
print('Rows:', len(r['data']), 'Elapsed seconds:', r['elapsed_seconds'])

## 9. AN7.05 — Resource breadth extraction (high cost)

Run only after AN7.04 has completed and been reviewed. This is a separate scan to isolate resource grouping cost.

## 10. Merge and reconciliation

Merge only on the full three-column key with one-to-one validation. No association is calculated until the AN6 totals reconcile.

In [ ]:
cohort=query_results['AN7.03']['data'].copy()
daily=query_results['AN7.04']['data'].copy()
breadth=query_results['AN7.05']['data'].copy()
for name,frame in [('cohort',cohort),('daily',daily),('breadth',breadth)]:
    assert len(frame)==EXPECTED_REGISTRATIONS, f'{name}: unexpected row count'
    assert not frame.duplicated(KEY).any(), f'{name}: duplicate key'
analysis_df=cohort.merge(daily,on=KEY,how='left',validate='one_to_one')
analysis_df=analysis_df.merge(breadth,on=KEY,how='left',validate='one_to_one')
assert len(analysis_df)==EXPECTED_REGISTRATIONS
assert analysis_df.id_student.nunique()==EXPECTED_DISTINCT_PEOPLE
assert int(analysis_df.course_active_days_full.gt(0).sum())==EXPECTED_ACTIVE_REGISTRATIONS
assert int(analysis_df.course_resource_breadth_full.gt(0).sum())==EXPECTED_ACTIVE_REGISTRATIONS
assert int(analysis_df.in_course_activity_rows_full.sum())==EXPECTED_IN_COURSE_ROWS
assert int(analysis_df.null_resource_key_groups.sum())==0

# Normalize active days by the number of days each learner could actually be observed after registration.
for landmark, day in [("L28",27),("L56",55),("L84",83)]:
    exposure = analysis_df[f"registered_exposure_days_{landmark.lower()}"]
    active = analysis_df[f"registered_active_days_{landmark.lower()}"]
    analysis_df[f"registered_active_day_rate_{landmark.lower()}"] = (
        active / exposure.where(exposure.gt(0))
    )
analysis_df["registered_active_day_rate_full"] = (
    analysis_df["registered_active_days_full"]
    / analysis_df["registered_exposure_days_full"].where(
        analysis_df["registered_exposure_days_full"].gt(0)
    )
)

pairs=[('course_active_days_l28','course_active_days_l56'),('course_active_days_l56','course_active_days_l84'),('course_active_days_l84','course_active_days_full'),('registered_active_days_l28','registered_active_days_l56'),('registered_active_days_l56','registered_active_days_l84'),('registered_active_days_l84','registered_active_days_full'),('course_resource_breadth_l28','course_resource_breadth_l56'),('course_resource_breadth_l56','course_resource_breadth_l84'),('course_resource_breadth_l84','course_resource_breadth_full'),('registered_resource_breadth_l28','registered_resource_breadth_l56'),('registered_resource_breadth_l56','registered_resource_breadth_l84'),('registered_resource_breadth_l84','registered_resource_breadth_full')]
for left,right in pairs:
    assert (analysis_df[left]<=analysis_df[right]).all(), f'Non-monotone: {left}>{right}'
print('AN7 extraction reconciliation: PASS')
print('Registrations:',len(analysis_df),'Distinct people:',analysis_df.id_student.nunique())
print('In-course active:',int(analysis_df.course_active_days_full.gt(0).sum()))
print('In-course source rows:',int(analysis_df.in_course_activity_rows_full.sum()))

## 11. Effect-size helpers

Binary outcomes use Cliff's delta. Confidence intervals resample student-presentations within each module-presentation; the grain contains one row per person within a presentation. Across-presentation results are not pooled as independent people.

In [ ]:
def cliffs_delta(x,y):
    x=pd.Series(x).dropna().to_numpy(); y=pd.Series(y).dropna().to_numpy()
    if not len(x) or not len(y): return np.nan
    u=mannwhitneyu(x,y,alternative='two-sided').statistic
    return float(2*u/(len(x)*len(y))-1)

def bootstrap_cliffs_ci(frame,feature,outcome_col,pos_label,neg_label,n_boot=300,seed=20261111):
    clean=frame[['id_student',feature,outcome_col]].dropna()
    if clean[outcome_col].nunique()!=2: return np.nan,np.nan,np.nan
    counts=clean[outcome_col].value_counts()
    if counts.get(pos_label,0)<10 or counts.get(neg_label,0)<10: return np.nan,np.nan,np.nan
    point=cliffs_delta(clean.loc[clean[outcome_col].eq(pos_label),feature],clean.loc[clean[outcome_col].eq(neg_label),feature])
    rng=np.random.default_rng(seed); boot=[]
    for _ in range(n_boot):
        sample=clean.iloc[rng.integers(0,len(clean),len(clean))]
        boot.append(cliffs_delta(sample.loc[sample[outcome_col].eq(pos_label),feature],sample.loc[sample[outcome_col].eq(neg_label),feature]))
    lo,hi=np.nanpercentile(boot,[2.5,97.5])
    return point,float(lo),float(hi)

LANDMARKS={'L28':'risk_status_l28','L56':'risk_status_l56','L84':'risk_status_l84'}
FEATURES={'active_day_rate':{'L28':'registered_active_day_rate_l28','L56':'registered_active_day_rate_l56','L84':'registered_active_day_rate_l84'},'resource_breadth':{'L28':'registered_resource_breadth_l28','L56':'registered_resource_breadth_l56','L84':'registered_resource_breadth_l84'}}
ORDINAL={'Fail':1,'Pass':2,'Distinction':3}

## 12. O1 and O2 — Binary associations by presentation and landmark

Positive Cliff's delta means the feature tends to be higher in the first-named outcome group. Confidence intervals are withheld when either outcome group has fewer than 10 observations; small cells remain visible in the descriptive output.

In [ ]:
binary_rows=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        eligible=group.loc[group[risk_col].eq('AT_RISK') & group.final_result.isin(['Withdrawn','Fail','Pass','Distinction'])].copy()
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]
            o1=eligible.copy(); o1['outcome']=np.where(o1.final_result.eq('Withdrawn'),'Withdrawn','Not_Withdrawn')
            delta,lo,hi=bootstrap_cliffs_ci(o1,feature,'outcome','Withdrawn','Not_Withdrawn')
            binary_rows.append({'outcome':'O1_WITHDRAWN','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n_positive':int(o1.outcome.eq('Withdrawn').sum()),'n_negative':int(o1.outcome.eq('Not_Withdrawn').sum()),'effect_size':delta,'ci_low':lo,'ci_high':hi})
            o2=eligible.loc[eligible.final_result.isin(['Fail','Pass','Distinction'])].copy()
            o2['outcome']=np.where(o2.final_result.eq('Fail'),'Fail','Pass_or_Distinction')
            delta,lo,hi=bootstrap_cliffs_ci(o2,feature,'outcome','Fail','Pass_or_Distinction')
            binary_rows.append({'outcome':'O2_FAIL','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n_positive':int(o2.outcome.eq('Fail').sum()),'n_negative':int(o2.outcome.eq('Pass_or_Distinction').sum()),'effect_size':delta,'ci_low':lo,'ci_high':hi})
binary_effects=pd.DataFrame(binary_rows)
display(binary_effects.head(20))

## 13. Descriptive distributions for O1 and O2

Effect sizes are interpreted alongside group distributions. This table reports sample sizes, medians, and interquartile ranges by outcome, feature, landmark, and module-presentation. Exposure-day summaries must also be reviewed because registration timing changes the available observation period.

In [ ]:
distribution_rows=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        eligible=group.loc[group[risk_col].eq('AT_RISK') & group.final_result.isin(['Withdrawn','Fail','Pass','Distinction'])].copy()
        eligible['O1']=np.where(eligible.final_result.eq('Withdrawn'),'Withdrawn','Not_Withdrawn')
        o2=eligible.loc[eligible.final_result.isin(['Fail','Pass','Distinction'])].copy()
        o2['O2']=np.where(o2.final_result.eq('Fail'),'Fail','Pass_or_Distinction')
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]
            for outcome_name,frame,outcome_col in [('O1',eligible,'O1'),('O2',o2,'O2')]:
                for group_name,part in frame.groupby(outcome_col):
                    values=part[feature].dropna()
                    distribution_rows.append({'outcome':outcome_name,'landmark':landmark,'code_module':module,
                        'code_presentation':pres,'feature':feature_name,'group':group_name,'n':len(values),
                        'median':values.median() if len(values) else np.nan,
                        'p25':values.quantile(.25) if len(values) else np.nan,
                        'p75':values.quantile(.75) if len(values) else np.nan})
outcome_distributions=pd.DataFrame(distribution_rows)
display(outcome_distributions.head(30))

## 14. O3 — Ordinal associations

In [ ]:
ordinal_rows=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        eligible=group.loc[group[risk_col].eq('AT_RISK') & group.final_result.isin(ORDINAL)].copy()
        eligible['outcome_ordinal']=eligible.final_result.map(ORDINAL)
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]; subset=eligible[['id_student',feature,'outcome_ordinal']].dropna()
            if len(subset)<20 or subset[feature].nunique()<2 or subset.outcome_ordinal.nunique()<2:
                rho,lo,hi=np.nan,np.nan,np.nan
            else:
                rho=float(spearmanr(subset[feature],subset.outcome_ordinal).statistic)
                rng=np.random.default_rng(20261111); boot=[]
                for _ in range(300):
                    sample=subset.iloc[rng.integers(0,len(subset),len(subset))]
                    if sample[feature].nunique()>1 and sample.outcome_ordinal.nunique()>1:
                        boot.append(spearmanr(sample[feature],sample.outcome_ordinal).statistic)
                lo,hi=np.nanpercentile(boot,[2.5,97.5]) if boot else (np.nan,np.nan)
            ordinal_rows.append({'outcome':'O3_ORDINAL','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n':len(subset),'spearman_rho':rho,'ci_low':lo,'ci_high':hi})
ordinal_effects=pd.DataFrame(ordinal_rows)
display(ordinal_effects.head(20))

## 15. Visual review of presentation-level associations

These boxplots summarize the distribution of presentation-specific effect estimates. Each presentation contributes one estimate; the figures are not pooled student-level effects and must not be read as a ranking of courses.

In [ ]:
for outcome_name, frame, value_col in [
    ('O1_WITHDRAWN', binary_effects, 'effect_size'),
    ('O2_FAIL', binary_effects, 'effect_size'),
    ('O3_ORDINAL', ordinal_effects, 'spearman_rho'),
]:
    for feature_name in FEATURES:
        subset=frame.loc[(frame.outcome.eq(outcome_name)) & frame.feature.eq(feature_name)].copy()
        if subset.empty:
            continue
        labels=[]; values=[]
        for landmark in LANDMARKS:
            series=subset.loc[subset.landmark.eq(landmark),value_col].dropna()
            if len(series):
                labels.append(landmark)
                values.append(series.to_numpy())
        if not values:
            continue
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.boxplot(values,labels=labels,showmeans=True)
        ax.axhline(0,linewidth=1)
        ax.set_title(f'{outcome_name}: presentation-level {feature_name} associations')
        ax.set_xlabel('Observation landmark')
        ax.set_ylabel(value_col)
        ax.grid(axis='y',alpha=.25)
        plt.tight_layout()
        plt.show()

## 16. Sensitivity analysis

Compare the primary O1 cohort with an alternative that treats undated Withdrawn cases as eligible. For O2, separately test exclusion of the nine Fail rows carrying an unregistration date. Source values are not relabelled.

In [ ]:
sensitivity=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        primary=group.loc[group[risk_col].eq('AT_RISK')].copy()
        alt=group.loc[group[risk_col].isin(['AT_RISK','INDETERMINATE_WITHDRAWAL_TIMING'])].copy()
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]
            for label,sample in [('O1_PRIMARY',primary),('O1_INCLUDE_UNDATED_WITHDRAWN',alt)]:
                sample=sample.loc[sample.final_result.isin(['Withdrawn','Fail','Pass','Distinction'])].copy()
                sample['outcome']=np.where(sample.final_result.eq('Withdrawn'),'Withdrawn','Not_Withdrawn')
                sensitivity.append({'sensitivity':label,'landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n':len(sample),'withdrawn_n':int(sample.outcome.eq('Withdrawn').sum()),'effect_size':cliffs_delta(sample.loc[sample.outcome.eq('Withdrawn'),feature],sample.loc[sample.outcome.eq('Not_Withdrawn'),feature])})
            o2=primary.loc[primary.final_result.isin(['Fail','Pass','Distinction'])].copy()
            o2['outcome']=np.where(o2.final_result.eq('Fail'),'Fail','Pass_or_Distinction')
            clean=o2.loc[~(o2.final_result.eq('Fail') & o2.date_unregistration.notna())]
            sensitivity.append({'sensitivity':'O2_EXCLUDE_FAIL_WITH_UNREGISTRATION_DATE','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n':len(clean),'fail_n':int(clean.outcome.eq('Fail').sum()),'effect_size':cliffs_delta(clean.loc[clean.outcome.eq('Fail'),feature],clean.loc[clean.outcome.eq('Pass_or_Distinction'),feature])})
sensitivity_effects=pd.DataFrame(sensitivity)
display(sensitivity_effects.head(20))

## 17. Interpretation checkpoint

Review group sizes, missingness, effect direction/magnitude, bootstrap interval width, presentation heterogeneity, and 93/9 sensitivities. Do not rank courses or pool registrations as independent people. No result is causal or predictive.

In [ ]:
print('SQL queries executed:',sorted(query_results))
print('Analysis rows:',len(analysis_df))
print('Binary effect rows:',len(binary_effects))
print('Ordinal effect rows:',len(ordinal_effects))
print('Sensitivity rows:',len(sensitivity_effects))

## 18. AN7 closeout is not automatic

AN7 may be closed only after returned SQL outputs, reconciliation checks, effect-size tables, sensitivity analyses, and the leakage register have been reviewed. Until local execution outputs are returned and interpreted, this notebook contains no findings.